# TrashSmart — huấn luyện bộ phận chai

Chọn Runtime → Change runtime type → GPU. Upload **training bundle** được tạo bằng `package-training`, không upload nguyên project có `.env`. Notebook dùng tài khoản Colab của bạn, không thuê GPU. Chỉ train từ dữ liệu đã rà soát; model ứng viên chưa tự bật trong website.


In [ ]:
%pip install -q ultralytics==8.4.164 Pillow==12.2.0


In [ ]:
from pathlib import Path
import json, zipfile, sys, torch
from google.colab import files
print("CUDA:",torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU hoặc dùng script CPU trên máy local")
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError("Chọn đúng một training bundle .zip")
archive=Path(next(iter(uploaded)))
ROOT=Path("/content/trashsmart-components").resolve()
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    if sum(i.file_size for i in z.infolist())>8*1024**3:
        raise ValueError("Bundle quá lớn")
    for entry in z.infolist():
        destination=(ROOT/entry.filename).resolve()
        if not destination.is_relative_to(ROOT):
            raise ValueError("Invalid archive path")
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT))
DATA=ROOT/"dataset"
report=json.loads((DATA/"dataset-report.json").read_text())
print(json.dumps(report["counts"],indent=2))


## Train detector nắp/thân/nhãn
Cần cả ba lớp trong train, val, test. Bắt đầu pilot vài epoch để đo tốc độ trước khi tăng. Ultralytics tải pretrained YOLO11n lần đầu. Đầu ra lưu riêng trong `/content`; tải về trước khi phiên bị thu hồi.


In [ ]:
from types import SimpleNamespace
from trashsmart.components.training import train_detector, train_material, evaluate
args=SimpleNamespace(data=DATA,output=ROOT/"candidate-detector",device="0",epochs=40,batch=16,imgsz=640,workers=2,threads=2,weights="yolo11n.pt",resume=None,split="test")
train_detector(args)


## Train vật liệu từ crop đã kiểm chứng
Bước này cần đủ glass/metal/paper/plastic trong train/val/test. Dữ liệu SPIN riêng không đáp ứng vì không có vật liệu được xác minh. ResNet18 baseline freeze backbone; có thể cần fine-tune thêm sau khi đánh giá. Không bỏ qua kiểm tra để chạy với nhãn đoán.


In [ ]:
material_ready=all(report["counts"].get(f"material/{split}/{material}",0)>=(4 if split=="train" else 1) for split in ["train","val","test"] for material in ["glass","metal","paper","plastic"])
if material_ready:
    material_args=SimpleNamespace(**vars(args))
    material_args.output=ROOT/"candidate-material"
    material_args.epochs=20
    material_args.batch=32
    train_material(material_args)
else:
    print("Bỏ qua classifier: chưa đủ nhãn vật liệu đã kiểm chứng. Chỉ train detector; chưa bật được nhận diện vật liệu bộ phận.")


## Đánh giá riêng, chưa phát hành
Report đo detector và classifier trên test; cần kiểm tra thêm toàn luồng trên ảnh chai thật. `endToEndReviewed` được giữ false cho đến khi thực sự rà soát.


In [ ]:
if material_ready:
    eval_args=SimpleNamespace(**vars(args))
    eval_args.output=ROOT/"evaluation"
    eval_args.weights=str(ROOT/"candidate-detector/detector/weights/best.pt")
    eval_args.resume=ROOT/"candidate-material/best.pth"
    evaluate(eval_args)
else:
    from ultralytics import YOLO
    model=YOLO(str(ROOT/"candidate-detector/detector/weights/best.pt"))
    metrics=model.val(data=str(ROOT/"candidate-detector/dataset.yaml"),split="test",device="0",project=str(ROOT/"evaluation"),name="detector-only")
    (ROOT/"evaluation/detector-only.json").write_text(json.dumps({"split":"test","detector":{k:float(v) for k,v in metrics.results_dict.items()},"materialClassifierReady":False,"activated":False},indent=2))


In [ ]:
import shutil
results=ROOT/"results"
results.mkdir(exist_ok=True)
for name in ["candidate-detector","candidate-material","evaluation"]:
    src=ROOT/name
    if src.exists():
        shutil.copytree(src,results/name,dirs_exist_ok=True)
bundle=shutil.make_archive("/content/bottle-components-results","zip",results)
files.download(bundle)


## Tiếp tục job bị gián đoạn
Giữ `last.pt`/`last.pth` trên máy hoặc Drive của bạn. Detector dùng resume chuẩn Ultralytics. Material dùng `--resume last.pth` và `--epochs` là tổng epoch. Sau khi tải kết quả về local, thực hiện quy trình đánh giá/phát hành trong `COMPONENTS.md`. Không tự đổi `validated=true` khi chỉ mới train xong.
